# 🇮🇩 Wicara 56M Chat — Interactive Demo

[![Hugging Face Model](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-num1notsvn%2Fwicara--56m--chat-blue)](https://huggingface.co/num1notsvn/wicara-56m-chat)
[![License](https://img.shields.io/badge/License-Apache--2.0-green.svg)](https://www.apache.org/licenses/LICENSE-2.0)

**Wicara** (*Weight-efficient Indonesian Conversational Architecture*) adalah model bahasa Indonesia 56 juta parameter yang dilatih dari nol (*from scratch*) dengan efisiensi komputasi tinggi oleh **Bagus Ardin**.

- **Arsitektur**: Gaya LLaMA (RMSNorm, RoPE, SwiGLU, GQA, Tied Embeddings)
- **Model Hub**: [num1notsvn/wicara-56m-chat](https://huggingface.co/num1notsvn/wicara-56m-chat)

---

### Langkah 1: Instalasi Library

In [ ]:
!pip install -q transformers gradio accelerate

### Langkah 2: Jalankan Chatbot Wicara

In [ ]:
import threading
import torch
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

MODEL_ID = "num1notsvn/wicara-56m-chat"

print(f"Memuat model {MODEL_ID} dari Hugging Face Hub...")
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype).to(device)
model.eval()
print(f"Model berhasil dimuat pada device: {device} ({dtype})!")


def _to_text(val) -> str:
    if isinstance(val, str):
        return val
    if isinstance(val, list):
        parts = []
        for part in val:
            if isinstance(part, str):
                parts.append(part)
            elif isinstance(part, dict):
                parts.append(str(part.get("text", "")))
            else:
                parts.append(str(part))
        return " ".join(parts)
    if isinstance(val, dict):
        return str(val.get("text", val.get("content", "")))
    return str(val)


def predict(
    message,
    history: list,
    system_prompt: str,
    temperature: float,
    top_p: float,
    repetition_penalty: float,
    max_new_tokens: int,
):
    user_text = _to_text(message).strip()
    if not user_text:
        yield ""
        return

    messages = []
    if system_prompt and system_prompt.strip():
        messages.append({"role": "system", "content": system_prompt.strip()})

    for item in history:
        if isinstance(item, dict):
            role = item.get("role", "user")
            content = _to_text(item.get("content", ""))
            if content:
                messages.append({"role": role, "content": content})
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            u, b = item
            u_text = _to_text(u)
            b_text = _to_text(b)
            if u_text:
                messages.append({"role": "user", "content": u_text})
            if b_text:
                messages.append({"role": "assistant", "content": b_text})

    messages.append({"role": "user", "content": user_text})

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=int(max_new_tokens),
        temperature=float(temperature),
        top_p=float(top_p),
        repetition_penalty=float(repetition_penalty),
        do_sample=True,
        eos_token_id=[tokenizer.eos_token_id, 2],
    )

    thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    partial_text = ""
    for new_text in streamer:
        partial_text += new_text
        yield partial_text


demo = gr.ChatInterface(
    fn=predict,
    title="🇮🇩 Wicara 56M Chat",
    description="Model bahasa Indonesia 56 juta parameter dilatih dari nol oleh Bagus Ardin.",
    additional_inputs=[
        gr.Textbox(
            value="Kamu adalah asisten AI berbahasa Indonesia yang ramah.",
            label="System Prompt",
            lines=2,
        ),
        gr.Slider(0.1, 1.5, value=0.7, step=0.05, label="Temperature"),
        gr.Slider(0.1, 1.0, value=0.9, step=0.05, label="Top-p"),
        gr.Slider(1.0, 1.5, value=1.15, step=0.05, label="Repetition Penalty"),
        gr.Slider(32, 256, value=160, step=16, label="Max New Tokens"),
    ],
    examples=[
        ["Halo! Siapa namamu?"],
        ["Apa kabar hari ini?"],
        ["Jelaskan cara membuat nasi goreng sederhana."],
        ["Ceritakan sesuatu yang menarik tentang Indonesia."],
    ],
    cache_examples=False,
)

# Di Google Colab, share=True akan otomatis menghasilkan public link gradio.live
demo.launch(share=True)